**AZ Watch** is a popular video streaming platform specialized in educational content, where creators publish online video tutorials and lessons about any topic, from speaking a new language to cooking to learning to play a musical instrument.

Their next goal is to leverage AI-driven solutions to analyze and make predictions about their subscribers and improve their marketing strategy around attracting new subscribers and retaining current ones. This project uses machine learning to predict subscribers likely to churn and find customer segments. This may help AZ Watch find interesting usage patterns to build subscriber personas in future marketing plans!

![Woman working on multiple screens](marketinganalytics.jpg)


The `data/AZWatch_subscribers.csv` **dataset** contains information about subscribers and their status over the last year:

|Column name|Description|
|-----------|-----------|
|`subscriber_id`|The unique identifier of each subscriber user|
|`age_group`|The subscriber's age group|
|`engagement_time`|Average time (in minutes) spent by the subscriber per session|
|`engagement_frequency`|Average weekly number of times the subscriber logged in the platform (sessions) over a year period|
|`subscription_status`|Whether the user remained subscribed to the platform by the end of the year period (subscribed), or unsubscribed and terminated her/his services (churned)|

Carefully observe and analyze the features in the dataset, asking yourself if there are any **categorical attributes** requiring pre-processing?

The subscribers dataset from the `data/AZWatch_subscribers.csv` file is already being loaded and split into training and test sets for you:

In [20]:
# Import the necessary modules
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.cluster import KMeans
import seaborn as sns
from matplotlib import pyplot as plt

# Specify the file path of your CSV file
file_path = "data/AZWatch_subscribers.csv"

# Read the CSV file into a DataFrame
df = pd.read_csv(file_path)

# Separate predictor variables from class label
X = df.drop(['subscriber_id','subscription_status'], axis=1)
y = df.subscription_status

# Split intro training and test sets (20% test)
X_train, X_test, y_train, y_test = train_test_split(
                        X, y, test_size=.2, random_state=42)

In [21]:
# Start your code here! Use as many cells as you like!
# Pre-processing: encode age_group (ordinal)
age_order = ['18-24', '25-34', '35-44', '45-54', '55+']
df['age_group'] = pd.Categorical(df['age_group'], categories=age_order, ordered=True)
df['age_group'] = df['age_group'].cat.codes

# Separate features and target
X = df.drop(['subscriber_id', 'subscription_status'], axis=1)
y = df['subscription_status']

# Encode target label
le = LabelEncoder()
y_enc = le.fit_transform(y)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Model 1 – Logistic Regression
model1 = LogisticRegression(max_iter=1000, random_state=42)
model1.fit(X_train_sc, y_train)
acc1 = accuracy_score(y_test, model1.predict(X_test_sc))

# Model 2 – Decision Tree
model2 = DecisionTreeClassifier(random_state=42)
model2.fit(X_train, y_train)
acc2 = accuracy_score(y_test, model2.predict(X_test))

# Model 3 – Random Forest
model3 = RandomForestClassifier(n_estimators=100, random_state=42)
model3.fit(X_train, y_train)
acc3 = accuracy_score(y_test, model3.predict(X_test))

print(f"Model 1 – Logistic Regression : {acc1:.4f}")
print(f"Model 2 – Decision Tree       : {acc2:.4f}")
print(f"Model 3 – Random Forest       : {acc3:.4f}")

# Best accuracy
score = max(acc1, acc2, acc3)
print(f"\nBest accuracy score: {score:.4f}")

# ── Clustering ────────────────────────────────────────────────
# Extract numerical features from X
segmentation = X[['engagement_time', 'engagement_frequency']].copy()

scaler_cl = StandardScaler()
X_scaled_cl = scaler_cl.fit_transform(segmentation)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
segmentation['cluster_id'] = kmeans.fit_predict(X_scaled_cl)
df['cluster_id'] = segmentation['cluster_id'].values

# Verifikation
print(df[['subscriber_id', 'cluster_id']].head())

# Cluster analysis
analysis = (df
    .groupby('cluster_id')[['engagement_time', 'engagement_frequency']]
    .mean()
    .round(0)
    .astype(int))
print("\nCluster Analysis:\n", analysis)

Model 1 – Logistic Regression : 0.9150
Model 2 – Decision Tree       : 0.8750
Model 3 – Random Forest       : 0.8950

Best accuracy score: 0.9150
   subscriber_id  cluster_id
0          14451           1
1          18386           2
2          12305           1
3          17546           2
4          15399           2

Cluster Analysis:
             engagement_time  engagement_frequency
cluster_id                                       
0                         9                     9
1                         4                     5
2                         7                    18
